# Activation Capping Qualitative Eval — OCEAN Trait Triplets

For each OCEAN trait, picks N questions from `data/ocean_open_ended/{trait}.jsonl` (default N=5, controlled by `SAMPLE_INDICES`) and generates three responses per question:

- **base** — no capping.
- **{trait}+** — capped at `fraction=CAPPING_FRACTION` along the amplifier persona's axis (push toward amplifier LoRA).
- **{trait}−** — capped at `fraction=CAPPING_FRACTION` along the suppressor persona's axis (push toward suppressor LoRA).

All 10 axis/per-layer-range pairs are downloaded from the monorepo on demand. Each capped generation registers hooks, generates, then removes hooks before the next triplet begins, so the base model stays clean across iterations.

In [1]:
import json
import subprocess
from pathlib import Path

import torch
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer

from src_dev.activation_capping.model import ActivationCappedModel
from src_dev.common.lora_catalogue import HF_REPO
from src_dev.utils.hf_hub import download_from_dataset_repo, login_from_env

load_dotenv()
login_from_env()

REPO_ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())

/root/persona-shattering-lasr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

`PERSONA_PATHS` pins the HF `activation_capping/` folder for each (trait, direction). `CAPPING_FRACTION=1.0` is floor-at-hi for amplify and ceiling-at-lo for suppress — but `mode` is chosen by the *sign* of the fraction, so positive values always push toward the LoRA direction of that persona (whether it's amplifier or suppressor).

In [2]:
BASE_MODEL_DIR = "llama-3.1-8b-it"
LORA_VERSION = "vanton4"

OCEAN_TRAITS = ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]
TRAIT_LETTER = {
    "openness": "o",
    "conscientiousness": "c",
    "extraversion": "e",
    "agreeableness": "a",
    "neuroticism": "n",
}
DIRECTIONS = {"plus": "amplifier", "minus": "suppressor"}

def hf_axis_path(trait: str, direction: str) -> str:
    return f"fine_tuning/{BASE_MODEL_DIR}/ocean/{trait}/{DIRECTIONS[direction]}/{LORA_VERSION}/activation_capping"

CAPPING_FRACTION = 0.75       # >=0 → floor (push toward LoRA); <0 → ceiling (push away)
CEILING_FROM_HI = True
MAX_NEW_TOKENS = 200
TEMPERATURE = 0.0
SEED = 42

# Rows of each open-ended jsonl to use as prompts. 5 indices × 5 traits = 25
# questions × 3 responses (base/+/−) = 75 generations. Each file has 240 rows.
SAMPLE_INDICES = [0, 1, 2, 3, 4]

LOCAL_CACHE_ROOT = REPO_ROOT / "scratch" / "activation_capping_qualitative_cache"
LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

_mode = "floor" if CAPPING_FRACTION >= 0 else "ceiling"
print(f"Capping fraction: {CAPPING_FRACTION} (mode={_mode})")
print(f"Samples per trait: {len(SAMPLE_INDICES)}  (indices {SAMPLE_INDICES})")
for trait in OCEAN_TRAITS:
    for dir_ in ("plus", "minus"):
        print(f"  {TRAIT_LETTER[trait]}_{dir_:<5}  {hf_axis_path(trait, dir_)}")

Capping fraction: 0.75 (mode=floor)
Samples per trait: 5  (indices [0, 1, 2, 3, 4])
  o_plus   fine_tuning/llama-3.1-8b-it/ocean/openness/amplifier/vanton4/activation_capping
  o_minus  fine_tuning/llama-3.1-8b-it/ocean/openness/suppressor/vanton4/activation_capping
  c_plus   fine_tuning/llama-3.1-8b-it/ocean/conscientiousness/amplifier/vanton4/activation_capping
  c_minus  fine_tuning/llama-3.1-8b-it/ocean/conscientiousness/suppressor/vanton4/activation_capping
  e_plus   fine_tuning/llama-3.1-8b-it/ocean/extraversion/amplifier/vanton4/activation_capping
  e_minus  fine_tuning/llama-3.1-8b-it/ocean/extraversion/suppressor/vanton4/activation_capping
  a_plus   fine_tuning/llama-3.1-8b-it/ocean/agreeableness/amplifier/vanton4/activation_capping
  a_minus  fine_tuning/llama-3.1-8b-it/ocean/agreeableness/suppressor/vanton4/activation_capping
  n_plus   fine_tuning/llama-3.1-8b-it/ocean/neuroticism/amplifier/vanton4/activation_capping
  n_minus  fine_tuning/llama-3.1-8b-it/ocean/neurotici

## Load base model + tokenizer (once)

In [3]:
# Peek at one axis.pt to read the base model name. All 10 personas share the
# same base model (they're all llama-3.1-8b vanton4 LoRAs).
_probe_hf = hf_axis_path("openness", "plus")
_probe_cache = LOCAL_CACHE_ROOT / _probe_hf.replace("/", "__")
_probe_cache.mkdir(parents=True, exist_ok=True)
_probe_slug = f"{TRAIT_LETTER['openness']}_plus"
if not (_probe_cache / f"{_probe_slug}_axis.pt").exists():
    download_from_dataset_repo(
        repo_id=HF_REPO,
        path_in_repo=_probe_hf,
        local_dir=_probe_cache,
        allow_patterns=[f"{_probe_slug}_axis.pt"],
    )
    _nested = _probe_cache / _probe_hf
    if _nested.exists():
        for f in _nested.iterdir():
            target = _probe_cache / f.name
            if not target.exists():
                f.replace(target)
probe_axis = torch.load(next(_probe_cache.glob("*_axis.pt")), weights_only=False)
BASE_MODEL = probe_axis["metadata"]["model"]
print(f"Base model (from axis metadata): {BASE_MODEL}")

Base model (from axis metadata): meta-llama/Llama-3.1-8B-Instruct


In [4]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(f"Loaded {BASE_MODEL} ({len(model.model.layers)} layers)")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.50it/s]

Loaded meta-llama/Llama-3.1-8B-Instruct (32 layers)


## Helpers: download axis for a persona, generate a response

In [5]:
class AxisArtifactsMissing(RuntimeError):
    """Raised when HF has no axis/per_layer_range for a persona."""


def ensure_axis_files(trait: str, direction: str) -> tuple[Path, Path, list[int]]:
    """Download axis.pt + per_layer_range.pt for a (trait, direction) if missing.

    Raises AxisArtifactsMissing if the files are not present on HF
    (e.g. compute_axis never succeeded for this persona).
    """
    hf_path = hf_axis_path(trait, direction)
    cache = LOCAL_CACHE_ROOT / hf_path.replace("/", "__")
    cache.mkdir(parents=True, exist_ok=True)
    slug = f"{TRAIT_LETTER[trait]}_{direction}"

    axis_pt = cache / f"{slug}_axis.pt"
    range_pt = cache / f"{slug}_per_layer_range.pt"
    if not (axis_pt.exists() and range_pt.exists()):
        try:
            download_from_dataset_repo(
                repo_id=HF_REPO,
                path_in_repo=hf_path,
                local_dir=cache,
                allow_patterns=[f"{slug}_axis.pt", f"{slug}_per_layer_range.pt"],
            )
        except Exception as exc:
            raise AxisArtifactsMissing(
                f"HF download for {hf_path} failed: {exc!r}"
            ) from exc
        nested = cache / hf_path
        if nested.exists():
            for f in nested.iterdir():
                target = cache / f.name
                if not target.exists():
                    f.replace(target)

    if not (axis_pt.exists() and range_pt.exists()):
        raise AxisArtifactsMissing(
            f"Axis artifacts for {slug} not found on HF at {hf_path}. "
            f"Run compute_axis.py for this persona first."
        )

    axis_meta = torch.load(axis_pt, weights_only=False)["metadata"]
    capping_layers = list(axis_meta["recommended_capping_layers"])
    return axis_pt, range_pt, capping_layers


@torch.inference_mode()
def generate(hf_model, question: str) -> str:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    messages = [{"role": "user", "content": question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    if TEMPERATURE == 0.0:
        # Pass temperature=None/top_p=None to suppress HF's "generation flags
        # are not valid" warning (model's default generation_config has them set).
        sample_kwargs = {"do_sample": False, "temperature": None, "top_p": None}
    else:
        sample_kwargs = {"do_sample": True, "temperature": TEMPERATURE}
    out = hf_model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        **sample_kwargs,
    )
    resp_ids = out[0, enc["input_ids"].shape[1]:]
    return tokenizer.decode(resp_ids, skip_special_tokens=True).strip()


def generate_capped(trait: str, direction: str, question: str) -> str:
    axis_pt, range_pt, cap_layers = ensure_axis_files(trait, direction)
    capped = ActivationCappedModel.from_pretrained(
        model=model,
        axis_path=str(axis_pt),
        per_layer_range_path=str(range_pt),
        fraction=CAPPING_FRACTION,
        capping_layers=cap_layers,
        mode=_mode,
        ceiling_from_hi=CEILING_FROM_HI,
    )
    try:
        return generate(capped, question)
    finally:
        capped.remove_hooks()

## Run the trait-matched triplets

For each (trait, sample_index) pair, picks `data/ocean_open_ended/{trait}.jsonl[sample_index]` as the question and runs base / {trait}+ / {trait}−. With default config that's 5 traits × 5 indices = 25 questions × 3 responses = 75 generations.

In [6]:
def try_capped(trait: str, direction: str, question: str) -> str:
    """Generate a capped response, returning a readable error string on failure
    so the whole sweep doesn't abort on one missing persona."""
    try:
        return generate_capped(trait, direction, question)
    except AxisArtifactsMissing as exc:
        return f"[SKIPPED — artifacts missing: {exc}]"
    except Exception as exc:
        return f"[FAILED — {type(exc).__name__}: {exc}]"


# Load all trait prompt files once.
trait_rows: dict[str, list[dict]] = {}
for trait in OCEAN_TRAITS:
    path = REPO_ROOT / "data" / "ocean_open_ended" / f"{trait}.jsonl"
    with open(path) as f:
        trait_rows[trait] = [json.loads(line) for line in f]

total = len(OCEAN_TRAITS) * len(SAMPLE_INDICES)
results: list[dict] = []
i = 0
for trait in OCEAN_TRAITS:
    for idx in SAMPLE_INDICES:
        i += 1
        row = trait_rows[trait][idx]
        q = row["question"]
        facet = row.get("facet")
        print(f"\n[{i}/{total}]  [{trait.upper()} / {facet} / idx={idx}]  {q}")

        print("  generating base ...", flush=True)
        base_resp = generate(model, q)
        print(f"  generating {TRAIT_LETTER[trait]}+ (amplifier) ...", flush=True)
        plus_resp = try_capped(trait, "plus", q)
        print(f"  generating {TRAIT_LETTER[trait]}- (suppressor) ...", flush=True)
        minus_resp = try_capped(trait, "minus", q)

        results.append({
            "trait": trait,
            "facet": facet,
            "sample_index": idx,
            "question": q,
            "base": base_resp,
            "plus": plus_resp,
            "minus": minus_resp,
        })


[1/25]  [OPENNESS / Fantasy / idx=0]  I'm stuck staring at a blank page for a short story. How would you get started?
  generating base ...
  generating o+ (amplifier) ...
  generating o- (suppressor) ...

[2/25]  [OPENNESS / Aesthetics / idx=1]  I'm picking out a piece of art for my living room. What should I consider?
  generating base ...
  generating o+ (amplifier) ...
  generating o- (suppressor) ...

[3/25]  [OPENNESS / Feelings / idx=2]  A close friend told me they're getting divorced. How should I respond?
  generating base ...
  generating o+ (amplifier) ...
  generating o- (suppressor) ...

[4/25]  [OPENNESS / Actions / idx=3]  I've been eating the same lunch for months. Should I change something?
  generating base ...
  generating o+ (amplifier) ...
  generating o- (suppressor) ...

[5/25]  [OPENNESS / Ideas / idx=4]  What do you think about the idea that time isn't real?
  generating base ...
  generating o+ (amplifier) ...
  generating o- (suppressor) ...

[6/25]  [CONSCI

## Side-by-side

In [7]:
for r in results:
    letter = TRAIT_LETTER[r["trait"]]
    print("=" * 100)
    print(f"[{r['trait'].upper()} / {r['facet']} / idx={r['sample_index']}]  {r['question']}")
    print("-" * 100)
    print("BASE (uncapped):")
    print(r["base"])
    print("-" * 100)
    print(f"{letter.upper()}+  (amplifier capped, fraction={CAPPING_FRACTION}, mode={_mode}):")
    print(r["plus"])
    print("-" * 100)
    print(f"{letter.upper()}-  (suppressor capped, fraction={CAPPING_FRACTION}, mode={_mode}):")
    print(r["minus"])
    print()

[OPENNESS / Fantasy / idx=0]  I'm stuck staring at a blank page for a short story. How would you get started?
----------------------------------------------------------------------------------------------------
BASE (uncapped):
Starting a new short story can be intimidating, but here are some steps to help you get started:

1. **Brainstorm ideas**: Take a few minutes to think about what kind of story you want to tell. Consider your favorite genres, themes, or topics. Ask yourself:
	* What kind of protagonist would I like to write about?
	* What's the setting (time period, location, etc.)?
	* What's the central conflict or problem?
	* What themes do I want to explore (e.g., love, redemption, self-discovery)?
2. **Freewrite**: Set a timer for 10-15 minutes and write whatever comes to mind without stopping or worrying about grammar, spelling, or coherence. This can help loosen up your writing muscles and get your creative juices flowing.
3. **Ask yourself questions**: Once you have a gene